Cell 1 — Imports and paths

In [14]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OrdinalEncoder, OneHotEncoder, FunctionTransformer
)
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from category_encoders import TargetEncoder

PROJECT_ROOT   = Path('..').resolve()
PROCESSED_DIR  = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR     = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)

df = pd.read_parquet(PROCESSED_DIR / 'loans_clean.parquet')
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Default rate: {df['target'].mean():.2%}")

Loaded: 1,345,350 rows × 34 columns
Default rate: 19.96%


Cell 2 — Temporal train/val/test split

In [15]:
df['issue_d'] = pd.to_datetime(df['issue_d'])
df['issue_year'] = df['issue_d'].dt.year

train = df[df['issue_year'] <= 2015].copy()
val   = df[df['issue_year'] == 2016].copy()
test  = df[df['issue_year'] >= 2017].copy()

print(f"Train : {train.shape[0]:>8,} rows  ({train['target'].mean():.2%} default)")
print(f"Val   : {val.shape[0]:>8,} rows  ({val['target'].mean():.2%} default)")
print(f"Test  : {test.shape[0]:>8,} rows  ({test['target'].mean():.2%} default)")
print(f"Total : {df.shape[0]:>8,} rows")

# Separate features and target — drop date cols (not used as model features)
DROP_COLS = ['issue_d', 'issue_year', 'target']

X_train = train.drop(columns=DROP_COLS)
y_train = train['target']

X_val   = val.drop(columns=DROP_COLS)
y_val   = val['target']

X_test  = test.drop(columns=DROP_COLS)
y_test  = test['target']

print(f"\nFeature matrix shape: {X_train.shape}")

Train :  826,606 rows  (18.43% default)
Val   :  293,105 rows  (23.29% default)
Test  :  225,639 rows  (21.29% default)
Total : 1,345,350 rows

Feature matrix shape: (826606, 31)


Cell 3 — Custom transformer: emp_length → integer months

In [16]:
class EmpLengthEncoder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        def parse(val):
            if pd.isnull(val):
                return -1
            val = str(val).strip().lower()
            if '10+' in val:
                return 10
            if '< 1' in val:
                return 0
            digits = ''.join(filter(str.isdigit, val.split('year')[0]))
            return int(digits) if digits else -1

        col = X.iloc[:, 0] if hasattr(X, 'iloc') else pd.Series(X.flatten())
        return col.map(parse).values.reshape(-1, 1)

    def get_feature_names_out(self, input_features=None):
        return np.array(['emp_length'])
        
# Quick smoke test
test_vals = pd.DataFrame({'emp_length': ['10+ years', '< 1 year', '5 years', np.nan, '2 years']})
enc = EmpLengthEncoder()
print("EmpLengthEncoder test:")
print(enc.fit_transform(test_vals[['emp_length']]).flatten())
# Expected: [10  0  5 -1  2]

EmpLengthEncoder test:
[10  0  5 -1  2]


Cell 4 — Custom transformer: log1p for annual_inc

In [17]:
log_transformer = FunctionTransformer(
    func=np.log1p,
    inverse_func=np.expm1,
    validate=True,
    feature_names_out='one-to-one'
)

# Test
sample = np.array([[50000], [120000], [0], [np.nan]])
# FunctionTransformer passes NaN through — the imputer upstream handles it
print("log1p transformer test (ignoring NaN for now):")
print(log_transformer.transform(np.array([[50000], [120000], [200000]])))

log1p transformer test (ignoring NaN for now):
[[10.81979828]
 [11.69525536]
 [12.20607765]]


Cell 5 — Define all feature groups

In [18]:
# ── Numerical: impute median → transform where needed ───────────────────────
NUM_LOG    = ['annual_inc']           # impute → log1p
NUM_PLAIN  = ['loan_amnt', 'int_rate', 'dti',
              'fico_range_low', 'revol_util', 'open_acc']  # impute only

# ── Ordinal: grade and sub_grade have a natural order ───────────────────────
ORD_GRADE     = ['grade']
ORD_SUBGRADE  = ['sub_grade']

GRADE_ORDER = [['A', 'B', 'C', 'D', 'E', 'F', 'G']]  # A=safest, G=riskiest
SUBGRADE_ORDER = [[
    'A1','A2','A3','A4','A5',
    'B1','B2','B3','B4','B5',
    'C1','C2','C3','C4','C5',
    'D1','D2','D3','D4','D5',
    'E1','E2','E3','E4','E5',
    'F1','F2','F3','F4','F5',
    'G1','G2','G3','G4','G5',
]]

# ── Target encoding: high-cardinality categorical ───────────────────────────
TARGET_ENC = ['addr_state']           # 51 states — target encode

# ── One-hot: low-cardinality categoricals ───────────────────────────────────
OHE_COLS   = ['home_ownership', 'purpose']

# ── Custom: emp_length needs string parsing first ───────────────────────────
EMP_COL    = ['emp_length']

# ── Drop: earliest_cr_line — convert to credit_age_months in next cell ──────
DATE_COL   = ['earliest_cr_line']

print("Feature groups defined:")
print(f"  Numerical (log)   : {NUM_LOG}")
print(f"  Numerical (plain) : {NUM_PLAIN}")
print(f"  Ordinal grade     : {ORD_GRADE}")
print(f"  Ordinal sub_grade : {ORD_SUBGRADE}")
print(f"  Target encoded    : {TARGET_ENC}")
print(f"  One-hot           : {OHE_COLS}")
print(f"  Custom emp_length : {EMP_COL}")
print(f"  Date (engineered) : {DATE_COL}")

Feature groups defined:
  Numerical (log)   : ['annual_inc']
  Numerical (plain) : ['loan_amnt', 'int_rate', 'dti', 'fico_range_low', 'revol_util', 'open_acc']
  Ordinal grade     : ['grade']
  Ordinal sub_grade : ['sub_grade']
  Target encoded    : ['addr_state']
  One-hot           : ['home_ownership', 'purpose']
  Custom emp_length : ['emp_length']
  Date (engineered) : ['earliest_cr_line']


Cell 6 — Engineer credit_age_months from earliest_cr_line

In [19]:
REFERENCE_DATE = pd.Timestamp('2018-12-31')  # end of dataset window

def add_credit_age(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()
    X['earliest_cr_line'] = pd.to_datetime(X['earliest_cr_line'], format='%b-%Y', errors='coerce')
    X['credit_age_months'] = (
        (REFERENCE_DATE - X['earliest_cr_line'])
        .dt.days / 30.44
    ).round(0).astype('float32')
    X = X.drop(columns=['earliest_cr_line'])
    return X

X_train = add_credit_age(X_train)
X_val   = add_credit_age(X_val)
X_test  = add_credit_age(X_test)

# Add to plain numericals list
NUM_PLAIN = NUM_PLAIN + ['credit_age_months']

print(f"credit_age_months sample: {X_train['credit_age_months'].describe()}")
print(f"Nulls after conversion  : {X_train['credit_age_months'].isnull().sum()}")

credit_age_months sample: count    826606.000000
mean        249.093216
std          89.154907
min          74.000000
25%         188.000000
50%         232.000000
75%         294.000000
max         900.000000
Name: credit_age_months, dtype: float64
Nulls after conversion  : 0


Cell 7 — Build the full ColumnTransformer pipeline


In [20]:
from sklearn.preprocessing import TargetEncoder as SklearnTargetEncoder

# ── Sub-pipelines ────────────────────────────────────────────────────────────

# 1. annual_inc: impute median → log1p
pipe_log = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('log',     log_transformer),
])

# 2. Plain numericals: impute median only
pipe_num = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])

# 3. grade: impute most_frequent → ordinal encode
pipe_grade = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(
        categories=GRADE_ORDER,
        handle_unknown='use_encoded_value',
        unknown_value=-1
    )),
])

# 4. sub_grade: impute most_frequent → ordinal encode
pipe_subgrade = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(
        categories=SUBGRADE_ORDER,
        handle_unknown='use_encoded_value',
        unknown_value=-1
    )),
])

# 5. addr_state: target encoding
pipe_target = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('target',  SklearnTargetEncoder(smooth='auto')),
])

# 6. One-hot: home_ownership, purpose
pipe_ohe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False,
        drop='first'
    )),
])

# 7. emp_length: custom string parser → impute median → pass through
pipe_emp = Pipeline([
    ('parser',  EmpLengthEncoder()),
    ('imputer', SimpleImputer(strategy='median')),
])

# ── ColumnTransformer: combine all sub-pipelines ─────────────────────────────
preprocessor = ColumnTransformer(
    transformers=[
        ('log_features',    pipe_log,      NUM_LOG),
        ('num_features',    pipe_num,      NUM_PLAIN),
        ('grade',           pipe_grade,    ORD_GRADE),
        ('subgrade',        pipe_subgrade, ORD_SUBGRADE),
        ('state_target',    pipe_target,   TARGET_ENC),
        ('ohe_features',    pipe_ohe,      OHE_COLS),
        ('emp_length',      pipe_emp,      EMP_COL),
    ],
    remainder='drop',
    verbose_feature_names_out=True
)

print("Preprocessor defined. Transformers:")
for name, pipe, cols in preprocessor.transformers:
    print(f"  {name:20s} → {cols}")

Preprocessor defined. Transformers:
  log_features         → ['annual_inc']
  num_features         → ['loan_amnt', 'int_rate', 'dti', 'fico_range_low', 'revol_util', 'open_acc', 'credit_age_months']
  grade                → ['grade']
  subgrade             → ['sub_grade']
  state_target         → ['addr_state']
  ohe_features         → ['home_ownership', 'purpose']
  emp_length           → ['emp_length']


Cell 8 — Fit the pipeline on training data only

In [21]:
print("Fitting preprocessor on training data...")

preprocessor.fit(X_train, y_train)

print("Fit complete.")

# Transform all three splits
X_train_proc = preprocessor.transform(X_train)
X_val_proc   = preprocessor.transform(X_val)
X_test_proc  = preprocessor.transform(X_test)

print(f"\nTransformed shapes:")
print(f"  X_train : {X_train_proc.shape}")
print(f"  X_val   : {X_val_proc.shape}")
print(f"  X_test  : {X_test_proc.shape}")

Fitting preprocessor on training data...
Fit complete.

Transformed shapes:
  X_train : (826606, 30)
  X_val   : (293105, 30)
  X_test  : (225639, 30)


Cell 9 — Recover and inspect feature names

In [22]:
feature_names = preprocessor.get_feature_names_out()

print(f"Total output features: {len(feature_names)}")
print("\nAll output feature names:")
for i, name in enumerate(feature_names):
    print(f"  [{i:2d}] {name}")

Total output features: 30

All output feature names:
  [ 0] log_features__annual_inc
  [ 1] num_features__loan_amnt
  [ 2] num_features__int_rate
  [ 3] num_features__dti
  [ 4] num_features__fico_range_low
  [ 5] num_features__revol_util
  [ 6] num_features__open_acc
  [ 7] num_features__credit_age_months
  [ 8] grade__grade
  [ 9] subgrade__sub_grade
  [10] state_target__addr_state
  [11] ohe_features__home_ownership_MORTGAGE
  [12] ohe_features__home_ownership_NONE
  [13] ohe_features__home_ownership_OTHER
  [14] ohe_features__home_ownership_OWN
  [15] ohe_features__home_ownership_RENT
  [16] ohe_features__purpose_credit_card
  [17] ohe_features__purpose_debt_consolidation
  [18] ohe_features__purpose_educational
  [19] ohe_features__purpose_home_improvement
  [20] ohe_features__purpose_house
  [21] ohe_features__purpose_major_purchase
  [22] ohe_features__purpose_medical
  [23] ohe_features__purpose_moving
  [24] ohe_features__purpose_other
  [25] ohe_features__purpose_renewable_en

Cell 10 — Sanity checks

In [23]:
# 1. No NaNs in output
assert not np.isnan(X_train_proc).any(), "NaNs found in X_train_proc!"
assert not np.isnan(X_val_proc).any(),   "NaNs found in X_val_proc!"
assert not np.isnan(X_test_proc).any(),  "NaNs found in X_test_proc!"
print("✓ No NaNs in any split")

# 2. annual_inc was log-transformed — check range
log_idx = list(feature_names).index('log_features__annual_inc')
log_vals = X_train_proc[:, log_idx]
print(f"✓ annual_inc log range: [{log_vals.min():.2f}, {log_vals.max():.2f}]")
print(f"  (original range was [{X_train['annual_inc'].min():,.0f}, "
      f"{X_train['annual_inc'].max():,.0f}])")

# 3. grade ordinal encoding — A=0, G=6
grade_idx = list(feature_names).index('grade__grade')
unique_grades = np.unique(X_train_proc[:, grade_idx])
print(f"✓ Grade ordinal values: {unique_grades}")

# 4. emp_length is numeric
emp_idx = [i for i, n in enumerate(feature_names) if 'emp' in n][0]
print(f"✓ emp_length range: [{X_train_proc[:, emp_idx].min():.0f}, "
      f"{X_train_proc[:, emp_idx].max():.0f}]")

# 5. class balance unchanged
print(f"✓ y_train default rate: {y_train.mean():.2%}")
print(f"✓ y_val   default rate: {y_val.mean():.2%}")
print(f"✓ y_test  default rate: {y_test.mean():.2%}")

print("\nAll sanity checks passed.")

✓ No NaNs in any split
✓ annual_inc log range: [0.00, 16.07]
  (original range was [0, 9,500,000])
✓ Grade ordinal values: [0. 1. 2. 3. 4. 5. 6.]
✓ emp_length range: [-1, 10]
✓ y_train default rate: 18.43%
✓ y_val   default rate: 23.29%
✓ y_test  default rate: 21.29%

All sanity checks passed.


Cell 11 — Convert to DataFrames and save processed splits

In [24]:
# Save as DataFrames with named columns — easier to work with in modeling notebook
X_train_df = pd.DataFrame(X_train_proc, columns=feature_names)
X_val_df   = pd.DataFrame(X_val_proc,   columns=feature_names)
X_test_df  = pd.DataFrame(X_test_proc,  columns=feature_names)

X_train_df.to_parquet(PROCESSED_DIR / 'X_train.parquet', index=False)
X_val_df.to_parquet(  PROCESSED_DIR / 'X_val.parquet',   index=False)
X_test_df.to_parquet( PROCESSED_DIR / 'X_test.parquet',  index=False)

y_train.reset_index(drop=True).to_frame().to_parquet(PROCESSED_DIR / 'y_train.parquet', index=False)
y_val.reset_index(drop=True).to_frame().to_parquet(  PROCESSED_DIR / 'y_val.parquet',   index=False)
y_test.reset_index(drop=True).to_frame().to_parquet( PROCESSED_DIR / 'y_test.parquet',  index=False)

print("Saved processed splits:")
print(f"  X_train : {X_train_df.shape}  →  data/processed/X_train.parquet")
print(f"  X_val   : {X_val_df.shape}  →  data/processed/X_val.parquet")
print(f"  X_test  : {X_test_df.shape}  →  data/processed/X_test.parquet")
print(f"  y splits: data/processed/y_{{train,val,test}}.parquet")

Saved processed splits:
  X_train : (826606, 30)  →  data/processed/X_train.parquet
  X_val   : (293105, 30)  →  data/processed/X_val.parquet
  X_test  : (225639, 30)  →  data/processed/X_test.parquet
  y splits: data/processed/y_{train,val,test}.parquet


Cell 12 — Save the fitted pipeline as .pkl

In [29]:
 
import cloudpickle
 
pipeline_path = MODELS_DIR / 'preprocessor.pkl'
with open(pipeline_path, 'wb') as f:
    cloudpickle.dump(preprocessor, f)
 
# Verify it loads back cleanly and produces identical output
with open(pipeline_path, 'rb') as f:
    preprocessor_loaded = cloudpickle.load(f)
 
X_check = preprocessor_loaded.transform(X_train.head(100))
import numpy as np
assert np.allclose(X_check, X_train_proc[:100]), "Reloaded pipeline output mismatch!"
 
print(f"Pipeline saved and verified: {pipeline_path}")
print(f"File size: {pipeline_path.stat().st_size / 1e3:.1f} KB")

Pipeline saved and verified: /Users/anuragpachgade/Desktop/finml/credit-default-prediction/models/preprocessor.pkl
File size: 9.7 KB


Cell 13 — Save metadata for inference time
At inference (Streamlit dashboard, FastAPI), you need to know exactly what the pipeline expects. This metadata file makes that explicit.

In [28]:
import json

metadata = {
    'feature_names_in': {
        'NUM_LOG'    : NUM_LOG,
        'NUM_PLAIN'  : NUM_PLAIN,
        'ORD_GRADE'  : ORD_GRADE,
        'ORD_SUBGRADE': ORD_SUBGRADE,
        'TARGET_ENC' : TARGET_ENC,
        'OHE_COLS'   : OHE_COLS,
        'EMP_COL'    : EMP_COL,
    },
    'feature_names_out'  : list(feature_names),
    'n_features_out'     : len(feature_names),
    'reference_date'     : str(REFERENCE_DATE.date()),
    'grade_order'        : GRADE_ORDER[0],
    'subgrade_order'     : SUBGRADE_ORDER[0],
    'train_default_rate' : float(y_train.mean()),
    'train_size'         : int(len(y_train)),
    'val_size'           : int(len(y_val)),
    'test_size'          : int(len(y_test)),
}

meta_path = MODELS_DIR / 'preprocessor_metadata.json'
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Metadata saved: {meta_path}")
print(json.dumps(metadata, indent=2)[:800], "...")

Metadata saved: /Users/anuragpachgade/Desktop/finml/credit-default-prediction/models/preprocessor_metadata.json
{
  "feature_names_in": {
    "NUM_LOG": [
      "annual_inc"
    ],
    "NUM_PLAIN": [
      "loan_amnt",
      "int_rate",
      "dti",
      "fico_range_low",
      "revol_util",
      "open_acc",
      "credit_age_months"
    ],
    "ORD_GRADE": [
      "grade"
    ],
    "ORD_SUBGRADE": [
      "sub_grade"
    ],
    "TARGET_ENC": [
      "addr_state"
    ],
    "OHE_COLS": [
      "home_ownership",
      "purpose"
    ],
    "EMP_COL": [
      "emp_length"
    ]
  },
  "feature_names_out": [
    "log_features__annual_inc",
    "num_features__loan_amnt",
    "num_features__int_rate",
    "num_features__dti",
    "num_features__fico_range_low",
    "num_features__revol_util",
    "num_features__open_acc",
    "num_features__credit_age_months",
    "grade__grade",
    "subgrade__sub_gra ...


In [30]:
import pickle

# Resave preprocessor with protocol 2 (most compatible across Python versions)
with open(MODELS_DIR / 'preprocessor.pkl', 'wb') as f:
    pickle.dump(preprocessor, f, protocol=2)

print("Done - preprocessor resaved with protocol 2")

Done - preprocessor resaved with protocol 2


Cell 14 — Commit checkpoint